# 🔧 Data Preprocessing & Feature Engineering

## IEEE Project: Phishing Guard v2.0

**Objective:** Extract 93 features from URLs for machine learning classification

### Feature Categories:
1. **Lexical Features** - URL structure analysis
2. **Host-Based Features** - Domain characteristics
3. **Security Features** - SSL/TLS analysis
4. **IDN Features** - Internationalized domain detection
5. **Composite Features** - Risk scoring

**Novel Contribution:** IDN/Homograph attack detection features

## 📚 Import Libraries

Import custom preprocessing modules from the project.

In [ ]:
import sys
import os
import yaml
import asyncio
import pandas as pd
import matplotlib.pyplot as plt

# Add utils to path
sys.path.append(os.path.abspath("../05_utils"))
from data_preparation import DataPreprocessor
from feature_extraction import URLFeatureExtractor

print("✅ Libraries imported successfully")
print(f"Python version: {sys.version}")

## ⚙️ Load Configuration

Configuration includes paths, feature extraction parameters, and processing settings.

In [ ]:
# Load configuration
with open("../07_configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("📋 Configuration loaded:")
print(f"  - Feature count: {config.get('features', {}).get('count', 'N/A')}")
print(f"  - Output directory: {config.get('paths', {}).get('processed_dir', 'N/A')}")

## 🚀 Initialize Preprocessor

The DataPreprocessor handles:
- Feature extraction
- Screenshot capture
- HTML content analysis
- Metadata generation

In [ ]:
# Initialize preprocessor
preprocessor = DataPreprocessor(config)
print("✅ Preprocessor initialized")
print(f"  - Extractor type: {type(preprocessor.extractor).__name__}")
print(f"  - Output directory: {preprocessor.output_dir}")

## 📊 Load Training Data

Load the training split for preprocessing.

In [ ]:
# Load training data
train_path = "../01_data/splits/train.csv"
train_df = pd.read_csv(train_path)

print(f"📊 Training data loaded: {len(train_df):,} samples")
print(f"\nLabel distribution:")
print(f"  Phishing: {sum(train_df['label'] == 1):,}")
print(f"  Legitimate: {sum(train_df['label'] == 0):,}")

# Display sample
print("\nSample data:")
train_df.head()

## 🔍 Test Feature Extraction (Single URL)

Let's test the feature extractor on a single URL to understand what features are extracted.

In [ ]:
# Test on a sample URL
test_url = "https://www.google.com"

print(f"🔍 Extracting features from: {test_url}\n")

# Extract features
features = preprocessor.extractor.extract_features(test_url)

print(f"✅ Extracted {len(features)} features\n")
print("Sample features (first 10):")
for i, (key, value) in enumerate(list(features.items())[:10]):
    print(f"  {key}: {value}")

## 📈 Feature Categories Visualization

Visualize the 93 features by category.

In [ ]:
# Categorize features
feature_categories = {
    'IDN/Unicode': 11,
    'Host Analysis': 10,
    'URL Patterns': 28,
    'Security': 6,
    'TLS/SSL': 11,
    'Composite': 3,
    'Other': 24
}

# Plot
plt.figure(figsize=(10, 6))
plt.bar(feature_categories.keys(), feature_categories.values(), 
        color=['#ef4444', '#3b82f6', '#10b981', '#f59e0b', '#8b5cf6', '#ec4899', '#6b7280'])
plt.title('Feature Distribution by Category (93 Total)', fontsize=16, fontweight='bold')
plt.xlabel('Category', fontsize=12)
plt.ylabel('Number of Features', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(f"\nTotal features: {sum(feature_categories.values())}")

## 🧪 Process Small Sample (Test Run)

Process a small sample first to verify everything works correctly.

In [ ]:
# Process small sample (first 10 URLs)
print("🔄 Processing test sample (10 URLs)...\n")

sample_df = train_df.head(10)
sample_csv = "../01_data/splits/temp_sample.csv"
sample_df.to_csv(sample_csv, index=False)

# Process
results = await preprocessor.process_dataset(
    csv_path=sample_csv,
    max_samples=10
)

print("\n✅ Processing complete!")
print(f"  Total: {len(results)}")
print(f"  Success: {sum(results['success'])}")
print(f"  Failed: {len(results) - sum(results['success'])}")

## 📊 Check Processing Results

Examine the processed features and output files.

In [ ]:
# Display results
print("Processing Results:")
print(results.head())

# Check output directories
processed_dir = os.path.abspath("../01_data/processed")
print(f"\n📁 Checking processed directory: {processed_dir}")

screenshots_dir = os.path.join(processed_dir, "screenshots")
html_dir = os.path.join(processed_dir, "html")
metadata_dir = os.path.join(processed_dir, "metadata")

print(f"  Screenshots: {len(os.listdir(screenshots_dir)) if os.path.exists(screenshots_dir) else 0} files")
print(f"  HTML files: {len(os.listdir(html_dir)) if os.path.exists(html_dir) else 0} files")
print(f"  Metadata: {len(os.listdir(metadata_dir)) if os.path.exists(metadata_dir) else 0} files")

## 💾 Save Processing Results

Save the feature extraction results for model training.

In [ ]:
# Save results
results_path = os.path.join(processed_dir, "processing_results.csv")
results.to_csv(results_path, index=False)

print(f"✅ Results saved to: {results_path}")
print(f"\nFile size: {os.path.getsize(results_path) / 1024:.1f} KB")

## 📊 Feature Statistics

Analyze the extracted features.

In [ ]:
# Display feature statistics
if 'features' in results.columns:
    print("Feature Statistics:")
    print(results['features'].describe())
else:
    print("Feature columns:")
    feature_cols = [col for col in results.columns if col not in ['url', 'label', 'success']]
    print(f"  Total feature columns: {len(feature_cols)}")
    print(f"\nFirst 10 features: {feature_cols[:10]}")

## 🎯 Summary

**Preprocessing Pipeline Results:**
- ✅ Successfully extracted features from {len(results)} URLs
- ✅ Generated {len(os.listdir(screenshots_dir)) if os.path.exists(screenshots_dir) else 0} screenshots
- ✅ Saved {len(os.listdir(html_dir)) if os.path.exists(html_dir) else 0} HTML snapshots
- ✅ Extracted 93 features per URL

**Key Features Implemented:**
1. **IDN Detection** - 11 features for homograph attack detection
2. **Host Analysis** - 10 features for domain characteristics
3. **URL Patterns** - 28 features for structural analysis
4. **Security/TLS** - 17 features for certificate validation
5. **Composite Scores** - 3 risk aggregation features

**Next Steps:**
1. Scale processing to full dataset
2. Handle failed URLs
3. Train/validation split
4. Model training with extracted features